In [0]:
drop table if exists demo.bronze.customer_ingest;

CREATE TABLE demo.bronze.customer_ingest
USING DELTA
AS
SELECT `Index` as index,	`Customer Id` as customer_id, `First Name` as first_name, `Last Name` as last_name, Company as company, City as city, Country as country, `Phone 1` as phone_1, `Phone 2` as phone_2, Email as email, `Subscription Date` as subscription_date, Website as website FROM read_files(
  '/Volumes/demo/bronze/data/customers-10000.csv',
  format => 'csv',
  header => true,
  inferSchema => true,
  mode => 'FAILFAST')

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists demo.bronze.product_ingest;

CREATE TABLE demo.bronze.product_ingest
USING DELTA
AS
SELECT Index as id, Index as product_id,	Name as product_name,	Description	as description, Brand	as brand, Category as category,	Price	as price, Currency as currency,	Stock	as stock, EAN	as ean, Color as color,	Size as size,	Availability as availability,	`Internal ID` as internal_id
FROM read_files(
  '/Volumes/demo/bronze/data/products-10000.csv',
  format => 'csv',
  header => true,
  inferSchema => true,
  mode => 'FAILFAST')

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists demo.bronze.order_ingest;

create table demo.bronze.order_ingest (
  order_id string not null,
  customer_id string not null,
  product_id int not null,
  order_date date,
  order_qty int,
  order_amount numeric(10,2),
  currency string,
  order_status string,
  order_channel string,
  ship_date date,
  ship_mode string,
  ship_cost numeric(10,2),
  remark string
);

insert into demo.bronze.order_ingest
select
  monotonically_increasing_id() as order_id,
  c.customer_id,
  p.product_id,
  date_add(current_date(), -cast(rand()*6 as int)) as order_date,
  cast(rand()*9 + 1 as int) as order_qty,
  cast(p.price * (rand()*9 + 1) as double) as order_amount,
  p.currency as currency,
  element_at(array('ordered','shipped','delivered','returned'), cast(rand()*3 + 1 as int)) as order_status,
  'online' as order_channel,
  date_add(current_date(), -cast(rand()*3 as int)) as ship_date,
  element_at(array('UPS','USPS','Fedex'), cast(rand()*2 + 1 as int)) as ship_mode,
  cast(rand()*9 + 1 as double) as ship_cost,
  element_at(array('ordinary shipping','priority shipping'), cast(rand()*1 + 1 as int)) as remark
from demo.bronze.customer_ingest c
inner join demo.bronze.product_ingest p
on c.index = p.id

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
drop table if exists demo.silver.h_customer;

create table if not exists demo.silver.h_customer
(
  customer_hk string not null comment "MD5(customer_id)",
  customer_id string NOT NULL,
  load_time timestamp not null,
  record_source string,
  constraint pk_h_customer primary key (customer_hk) 
) USING DELTA;


insert into demo.silver.h_customer
select 
  md5(customer_id) as customer_hk,
  customer_id,
  current_timestamp() as load_time,
  'customer-10000.csv' as record_source
from (
  select distinct customer_id from demo.bronze.customer_ingest
);

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
drop table if exists demo.silver.s_customer;

create table demo.silver.s_customer(
  customer_hk string not null comment "MD5(customer_id)",
  hash_diff string not null comment "Hash of all columns",
  customer_id string not null,
  first_name string not null,
  last_name string not null,
  company string not null,
  city string not null,
  country string not null,
  phone_1 string not null,
  phone_2 string not null,
  email string not null,
  subscription_date string not null,
  website string not null,
  load_time timestamp not null,
  record_source string,
  constraint pk_s_customer primary key(customer_hk)
) USING DELTA;


insert into demo.silver.s_customer 
select 
  md5(customer_id) as customer_hk,
  sha2(concat_ws(',',customer_id,first_name,last_name,company,city,country,phone_1,phone_2,email,subscription_date,website),256) as hash_diff,
  customer_id,
  first_name,
  last_name,
  company,
  city, 
  country,
  phone_1,
  phone_2,
  email,
  subscription_date,
  website,
  current_timestamp() as load_time,
  'customer-10000.csv' as record_source
from (
  select distinct customer_id, first_name, last_name, company, city, country, phone_1, phone_2, email, subscription_date, website 
  from demo.bronze.customer_ingest
)

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
drop table if exists demo.silver.h_product;

create table if not exists demo.silver.h_product
(
  product_hk string not null comment "MD5(product_id)",
  product_id INT NOT NULL,
  load_time timestamp not null,
  record_source string,
  constraint pk_h_product primary key (product_hk) 
) USING DELTA;

insert into demo.silver.h_product
select 
  md5(cast(product_id as string)) as product_hk,
  product_id,
  current_timestamp() as load_time,
  'product-10000.csv' as record_source
from (
  select distinct product_id from demo.bronze.product_ingest
);

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
drop table if exists demo.silver.s_product;

create table  demo.silver.s_product(
  product_hk string not null comment "MD5(product_id)",
  hash_diff string not null comment "Hash of all non-key columns",
  product_id INT not null,
  name string not null,
  description string not null,
  brand string not null,
  category string not null,
  price string not null,
  load_time timestamp not null,
  record_source string,
  constraint pk_s_product primary key(product_hk)
) USING DELTA;

insert into demo.silver.s_product
select
  md5(cast(product_id as string)) as product_hk,
  sha2(concat_ws(',', product_id, name, description, brand, category, price), 256) as hash_diff,
  product_id,
  name,
  description,
  brand,
  category,
  price,
  current_timestamp() as load_time,
  'product-10000.csv' as record_source
from (
  select distinct product_id, name, description, brand, category, price
  from demo.bronze.product_ingest
);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8488455278809567>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'drop table if exists demo.silver.s_product;\n\ncreate table  demo.silver.s_product(\n  product_hk string not null comment "MD5(product_id)",\n  hash_diff string not null comment "Hash of all non-key columns",\n  product_id INT not null,\n  name string not null,\n  description string not null,\n  brand string not null,\n  category string not null,\n  price string not null,\n  load_time timestamp not null,\n  record_source string,\n  constraint pk_s_product primary key(product_hk)\n) USING DELTA;\n\ninsert into demo.silver.s_product\nselect\n  md5(cast(product_id as string)) as product_hk,\n  sha2(concat_ws(\',\', product_id, name, description, brand, category, price), 256) as hash_diff,\n  product_id,\n  name,\n  description,\n  brand,\n  category,\n

In [0]:
%sql
drop table if exists demo.silver.l_order;

create table demo.silver.l_order
(
  order_hk string not null comment "MD5(order_id || customer_id || product_id)", 
  order_id INT not null,
  customer_id string NOT NULL,
  product_id INT NOT NULL,
  load_time timestamp not null,
  record_source string,
  constraint pk_order primary key(order_hk)
) using delta;

insert into demo.silver.l_order
select 
  md5(concat(o.order_id, o.customer_id, o.product_id)) as order_hk,
  o.order_id,
  o.customer_id,
  o.product_id,
  current_timestamp() as load_time,
  'system' as record_source
from demo.bronze.order_ingest o
WHERE o.order_id is not null;

In [0]:
%sql
drop table if exists demo.silver.s_l_order;

create table demo.silver.s_l_order
(
  order_hk string not null comment "MD5(order_id || customer_id || product_id)", 
  hash_diff string not null comment "Hash of all columns",
  order_date date not null,
  order_qty int not null,
  order_amount numeric(10,2) not null,
  currency string not null,
  order_status string not null,
  order_channel string not null,
  ship_date date not null,
  ship_mode string not null,
  ship_cost numeric(10,2) not null,
  remark string not null,
  load_time timestamp not null,
  record_source string,
  constraint pk_s_l_order primary key(order_hk)
) using delta;

insert into demo.silver.s_l_order
select
  md5(concat(order_id, customer_id, product_id)) as order_hk,
  sha2(concat_ws(',', order_date, order_qty, order_amount, currency, order_status, order_channel, ship_date, ship_mode, ship_cost, remark), 256) as hash_diff,
  order_date,
  order_qty,
  order_amount,
  currency,
  order_status,
  order_channel,
  ship_date,
  ship_mode,
  ship_cost,
  remark,
  current_timestamp() as load_time,
  'system' as record_source
from demo.bronze.order_ingest;


In [0]:
DROP VIEW IF EXISTS demo.gold.fact_order_view;

CREATE OR REPLACE VIEW demo.gold.fact_order_view
AS
SELECT
  lo.order_id,
  lo.customer_id,
  lo.product_id,
  slo.order_date,
  slo.order_amount,
  slo.order_status,
  slo.order_channel,
  slo.ship_date,
  slo.ship_mode,
  slo.ship_cost,
  slo.remark,
  slo.load_time,
  slo.record_source
FROM demo.silver.l_order lo 
JOIN demo.silver.s_l_order slo ON lo.order_hk = slo.order_hk
WHERE slo.ship_date >= slo.order_date;

DROP VIEW IF EXISTS demo.gold.dim_customer_view;

CREATE OR REPLACE VIEW demo.gold.dim_customer_view
AS
SELECT
  ov.customer_id,
  sc.first_name,
  sc.last_name,
  sc.company,
  sc.city,
  sc.country,
  sc.phone_1,
  sc.phone_2,
  sc.email,
  sc.subscription_date,
  sc.website,
  sc.load_time,
  sc.record_source
FROM demo.gold.fact_order_view ov
join demo.silver.h_customer hc on ov.customer_id = hc.customer_id
join demo.silver.s_customer sc on hc.customer_hk = sc.customer_hk;

DROP VIEW IF EXISTS demo.gold.dim_product_view;

CREATE OR REPLACE VIEW demo.gold.dim_product_view
AS
SELECT
  ov.product_id,
  sp.name,
  sp.description,
  sp.brand,
  sp.category,
  sp.price,
  sp.load_time,
  sp.record_source
FROM demo.gold.fact_order_view ov
join demo.silver.h_product hp on ov.product_id = hp.product_id
join demo.silver.s_product sp on hp.product_hk = sp.product_hk;